# EvoCRM v2 — Master Notebook

One self-contained notebook. No external package. All code inline. Runs end-to-end on Olist.

**Architecture:** Perceiver IO hub · FT-Transformer customer tower · output-query task heads · LoRA transfer

**Tasks:** churn (binary) · next category (multi-class) · CLV (regression)

**Sections:**
1. Setup and config
2. Data loading (Olist or synthetic fallback)
3. Leak-free labels
4. Features (cutoff-gated)
5. FT-Transformer (Customer Tower)
6. Perceiver IO hub + Output-query heads
7. Three input towers
8. Full EvoCRM model
9. PyTorch Dataset
10. Training loop
11. Evaluation
12. Specialist baselines
13. LoRA injection (self-evolving mechanism)
14. End-to-end run on Olist
15. Diagnostics (leak check on results)

## 1. Setup and config

In [ ]:
# Install if needed (uncomment if running in a fresh environment)
# !pip install torch pandas numpy scikit-learn --quiet

import os, sys, math, copy, json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.ensemble import (
    GradientBoostingClassifier, GradientBoostingRegressor,
    RandomForestClassifier, RandomForestRegressor,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    mean_squared_error, mean_absolute_error,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch {torch.__version__}  |  device: {DEVICE}')

In [ ]:
# ============================================================
# CONFIG — edit these for your run
# ============================================================
OLIST_DIR = './olist_raw'                   # path to Kaggle Olist CSVs
USE_SYNTHETIC_IF_MISSING = True             # fall back to synthetic if dir not found

OBSERVATION_WINDOW_DAYS = 90
MIN_FEATURE_WINDOW_DAYS = 30
MAX_SEQUENCE_LENGTH = 50
N_CATEGORY_CLASSES = 10                     # top-K categories + 'other'

# Model
FT_D_TOKEN = 64
FT_N_BLOCKS = 3
FT_N_HEADS = 8
PERCEIVER_NUM_LATENTS = 64
PERCEIVER_LATENT_DIM = 128
PERCEIVER_INPUT_DIM = 128
PERCEIVER_N_LAYERS = 4

# Training
BATCH_SIZE = 128
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
GRADIENT_CLIP = 1.0
PATIENCE = 5
VAL_FRACTION = 0.2

# LoRA
LORA_RANK = 8
LORA_ALPHA = 16

## 2. Data loading — Olist or synthetic fallback

Loads four canonical tables: `orders`, `order_items`, `products`, `customers`.

In [ ]:
def load_olist(root: str) -> Dict[str, pd.DataFrame]:
    root = Path(root)
    files = {
        'orders':      'olist_orders_dataset.csv',
        'order_items': 'olist_order_items_dataset.csv',
        'products':    'olist_products_dataset.csv',
        'customers':   'olist_customers_dataset.csv',
    }
    tables = {}
    for key, fname in files.items():
        p = root / fname
        if not p.exists():
            raise FileNotFoundError(f'Missing {p}')
        tables[key] = pd.read_csv(p)
    # Keep only the columns we need for each table
    tables['orders'] = tables['orders'][[
        'order_id', 'customer_id', 'order_purchase_timestamp', 'order_status',
    ]]
    tables['order_items'] = tables['order_items'][[
        'order_id', 'product_id', 'price', 'freight_value',
    ]]
    tables['products'] = tables['products'][[
        'product_id', 'product_category_name',
    ]]
    tables['customers'] = tables['customers'][[
        'customer_id', 'customer_unique_id',
    ]]
    return tables


def generate_synthetic_olist(
    n_customers=2000, n_products=200, n_categories=12,
    start='2022-01-01', end='2023-12-31', seed=SEED,
) -> Dict[str, pd.DataFrame]:
    """Poisson-rate per customer → realistic churn signal."""
    rng = np.random.default_rng(seed)
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    total_days = (end - start).days
    rates = np.clip(rng.lognormal(-3.5, 1.2, n_customers), 1e-4, 0.5)
    cat_names = [f'cat_{i:02d}' for i in range(n_categories)]
    product_ids = [f'prod_{i:05d}' for i in range(n_products)]
    product_cats = rng.choice(cat_names, size=n_products)
    product_prices = rng.lognormal(3.5, 0.7, n_products)
    orders, items, custs = [], [], []
    counter = 0
    for ci, rate in enumerate(rates):
        cuid = f'cust_{ci:06d}'
        n = max(rng.poisson(rate * total_days), 0)
        if n == 0:
            custs.append({'customer_id': f'cid_{ci:06d}_0', 'customer_unique_id': cuid})
            continue
        ts = sorted([start + pd.Timedelta(days=float(rng.uniform(0, total_days))) for _ in range(n)])
        for k, t in enumerate(ts):
            cid = f'cid_{ci:06d}_{k}'
            oid = f'ord_{counter:08d}'; counter += 1
            custs.append({'customer_id': cid, 'customer_unique_id': cuid})
            status = rng.choice(['delivered', 'shipped', 'canceled'], p=[0.92, 0.05, 0.03])
            orders.append({
                'order_id': oid, 'customer_id': cid,
                'order_purchase_timestamp': t, 'order_status': status,
            })
            for _ in range(rng.integers(1, 5)):
                pi = rng.integers(0, n_products)
                items.append({
                    'order_id': oid, 'product_id': product_ids[pi],
                    'price': float(product_prices[pi]),
                    'freight_value': float(rng.uniform(5, 25)),
                })
    return {
        'orders':      pd.DataFrame(orders),
        'order_items': pd.DataFrame(items),
        'products':    pd.DataFrame({'product_id': product_ids, 'product_category_name': product_cats}),
        'customers':   pd.DataFrame(custs).drop_duplicates('customer_id'),
    }


# Decide which data source to use
if Path(OLIST_DIR).exists() and (Path(OLIST_DIR) / 'olist_orders_dataset.csv').exists():
    print(f'Loading REAL Olist data from {OLIST_DIR}...')
    tables = load_olist(OLIST_DIR)
    DATA_SOURCE = 'olist'
elif USE_SYNTHETIC_IF_MISSING:
    print(f'Olist not found at {OLIST_DIR}. Falling back to SYNTHETIC data.')
    print('To use real data: download from https://www.kaggle.com/olistbr/brazilian-ecommerce')
    tables = generate_synthetic_olist(n_customers=3000)
    DATA_SOURCE = 'synthetic'
else:
    raise FileNotFoundError(f'Olist not at {OLIST_DIR} and USE_SYNTHETIC_IF_MISSING=False')

for name, df in tables.items():
    print(f'  {name:>14s}: {df.shape}')

## 3. Leak-free labels

**Temporal layout:**
```
  |<------ feature window ------>|<-- observation (90d) -->|
  dataset_start              cutoff_date           cutoff + 90d
```

Features use data `<= cutoff_date`. Labels use data `> cutoff_date` only. No overlap.

In [ ]:
def suggest_cutoff(orders, observation_days=OBSERVATION_WINDOW_DAYS):
    ts = pd.to_datetime(orders['order_purchase_timestamp'])
    return ts.max().normalize() - pd.Timedelta(days=observation_days)


def build_category_labels(obs_orders, order_items, products, eligible, top_k=N_CATEGORY_CLASSES):
    """Category of first post-cutoff order. -1 = no next order (invalid label)."""
    if len(obs_orders) == 0:
        return pd.Series(-1, index=eligible, dtype=int)
    first_obs = (obs_orders.sort_values('order_purchase_timestamp')
                           .groupby('customer_unique_id').first().reset_index())
    first_items = first_obs.merge(order_items[['order_id', 'product_id']], on='order_id', how='left')
    first_items = first_items.merge(products[['product_id', 'product_category_name']],
                                    on='product_id', how='left')
    first_per = first_items.groupby('customer_unique_id').first()
    vc = first_per['product_category_name'].value_counts()
    top = vc.head(top_k - 1).index.tolist()
    cmap = {c: i for i, c in enumerate(top)}
    def mp(c):
        if pd.isna(c): return -1
        return cmap.get(c, top_k - 1)   # 'other' bucket
    cats = first_per['product_category_name'].apply(mp)
    return cats.reindex(eligible, fill_value=-1).astype(int)


def build_all_labels(orders, order_items, products, customers, cutoff_date):
    """Build churn (binary) + category (multi-class) + CLV (regression) labels."""
    cutoff = pd.Timestamp(cutoff_date)
    obs_end = cutoff + pd.Timedelta(days=OBSERVATION_WINDOW_DAYS)

    # Resolve customer_unique_id (Olist: customer_id changes per order)
    o = orders.merge(customers[['customer_id', 'customer_unique_id']],
                     on='customer_id', how='inner', validate='many_to_one')
    o['order_purchase_timestamp'] = pd.to_datetime(o['order_purchase_timestamp'])
    o = o[~o['order_status'].isin(['canceled', 'unavailable'])].copy()

    dataset_max = o['order_purchase_timestamp'].max()
    if obs_end > dataset_max:
        raise ValueError(f'Observation window {obs_end.date()} > dataset_max {dataset_max.date()}')

    in_feat = o['order_purchase_timestamp'] <= cutoff
    in_obs = (o['order_purchase_timestamp'] > cutoff) & (o['order_purchase_timestamp'] <= obs_end)
    feat, obs = o.loc[in_feat], o.loc[in_obs]

    # Eligibility
    feat_agg = feat.groupby('customer_unique_id').agg(
        n=('order_id', 'nunique'),
        first=('order_purchase_timestamp', 'min'),
    )
    min_win = pd.Timedelta(days=MIN_FEATURE_WINDOW_DAYS)
    eligible = feat_agg[(feat_agg['n'] >= 1) & ((cutoff - feat_agg['first']) >= min_win)].index

    # Churn
    obs_counts = obs.groupby('customer_unique_id')['order_id'].nunique().reindex(eligible, fill_value=0)
    churn = (obs_counts == 0).astype(int)

    # CLV
    obs_items = obs.merge(order_items, on='order_id', how='inner')
    obs_items['line_total'] = obs_items['price'] + obs_items['freight_value']
    clv = obs_items.groupby('customer_unique_id')['line_total'].sum().reindex(eligible, fill_value=0.0)

    # Category
    category = build_category_labels(obs, order_items, products, eligible)

    labels = pd.DataFrame({
        'churn': churn.values,
        'category': category.values,
        'clv': clv.values,
    }, index=eligible)
    labels['churn_valid'] = True
    labels['category_valid'] = category.values >= 0
    labels.loc[labels['churn'] == 1, 'category_valid'] = False   # churners have no next order
    labels['clv_valid'] = True
    labels.index.name = 'customer_unique_id'
    return labels.reset_index()


# Build labels
cutoff = suggest_cutoff(tables['orders'])
print(f'cutoff_date: {cutoff}')
labels = build_all_labels(tables['orders'], tables['order_items'],
                          tables['products'], tables['customers'], cutoff)
print(f'\nn labeled customers: {len(labels):,}')
print(f'churn rate:          {labels["churn"].mean():.3f}')
print(f'category validity:   {labels["category_valid"].mean():.3f}')
print(f'mean CLV (obs):      {labels["clv"].mean():.2f}')

# INVARIANTS
assert labels.loc[labels['churn']==1, 'clv'].max() == 0, 'LEAK: churner has CLV>0'
assert (labels.loc[labels['churn']==1, 'category_valid']==False).all(), 'churner should have invalid cat'
print('\n✓ Label invariants hold')
labels.head()

## 4. Features — cutoff-gated

Every aggregation filters source data to `<= cutoff` BEFORE computing anything. Invariant: `recency_days >= 0` always.

In [ ]:
def build_customer_features(orders, order_items, customers, cutoff_date):
    cutoff = pd.Timestamp(cutoff_date)
    o = orders.merge(customers[['customer_id', 'customer_unique_id']],
                     on='customer_id', how='inner', validate='many_to_one')
    o['order_purchase_timestamp'] = pd.to_datetime(o['order_purchase_timestamp'])
    o = o[o['order_purchase_timestamp'] <= cutoff]   # <-- CUTOFF GUARD
    o = o[~o['order_status'].isin(['canceled', 'unavailable'])]

    if len(o) == 0:
        cols = ['recency_days','frequency','monetary','avg_order_value','tenure_days','n_items']
        return pd.DataFrame(columns=cols).rename_axis('customer_unique_id')

    items = o.merge(order_items, on='order_id', how='inner')
    items['line_total'] = items['price'] + items['freight_value']
    order_totals = items.groupby(['customer_unique_id', 'order_id'])['line_total'].sum().reset_index()

    a_ord = o.groupby('customer_unique_id').agg(
        last=('order_purchase_timestamp', 'max'),
        first=('order_purchase_timestamp', 'min'),
        frequency=('order_id', 'nunique'),
    )
    a_money = order_totals.groupby('customer_unique_id').agg(
        monetary=('line_total', 'sum'),
        avg_order_value=('line_total', 'mean'),
    )
    a_items = items.groupby('customer_unique_id').agg(n_items=('product_id', 'count'))

    feats = a_ord.join(a_money).join(a_items)
    feats['recency_days'] = (cutoff - feats['last']).dt.days
    feats['tenure_days'] = (cutoff - feats['first']).dt.days
    feats = feats.drop(columns=['last', 'first'])
    return feats[['recency_days','frequency','monetary','avg_order_value','tenure_days','n_items']].fillna(0)


@dataclass
class SequenceData:
    customer_ids: np.ndarray
    product_ids: np.ndarray
    timestamps_delta: np.ndarray
    mask: np.ndarray
    product_id_vocab_size: int


def build_interaction_sequences(orders, order_items, customers, cutoff_date,
                                max_length=MAX_SEQUENCE_LENGTH):
    cutoff = pd.Timestamp(cutoff_date)
    o = orders.merge(customers[['customer_id', 'customer_unique_id']],
                     on='customer_id', how='inner', validate='many_to_one')
    o['order_purchase_timestamp'] = pd.to_datetime(o['order_purchase_timestamp'])
    o = o[o['order_purchase_timestamp'] <= cutoff].copy()
    o = o[~o['order_status'].isin(['canceled', 'unavailable'])]
    items = o.merge(order_items, on='order_id', how='inner')
    items = items.sort_values(['customer_unique_id', 'order_purchase_timestamp'])

    pid_to_idx = {p: i+1 for i, p in enumerate(items['product_id'].unique())}  # 0 = pad
    vocab = len(pid_to_idx) + 1

    cids, pids, dts, masks = [], [], [], []
    for cuid, grp in items.groupby('customer_unique_id', sort=False):
        p = grp['product_id'].map(pid_to_idx).to_numpy()
        ts = grp['order_purchase_timestamp'].to_numpy()
        d = ((cutoff - pd.to_datetime(ts)).total_seconds().to_numpy() / 86400.0).astype(float)
        if len(p) > max_length:
            p = p[-max_length:]; d = d[-max_length:]
        L = len(p); pad = max_length - L
        cids.append(cuid)
        pids.append(np.concatenate([p, np.zeros(pad, dtype=np.int64)]))
        dts.append(np.concatenate([d, np.zeros(pad, dtype=np.float32)]))
        masks.append(np.concatenate([np.ones(L, dtype=bool), np.zeros(pad, dtype=bool)]))

    if not cids:
        return SequenceData(
            np.array([]), np.zeros((0, max_length), np.int64),
            np.zeros((0, max_length), np.float32), np.zeros((0, max_length), bool), vocab,
        )
    return SequenceData(
        np.array(cids),
        np.stack(pids).astype(np.int64),
        np.stack(dts).astype(np.float32),
        np.stack(masks),
        vocab,
    )


def build_customer_category_ids(orders, order_items, products, customers, cutoff_date,
                                customer_ids, top_k=3):
    cutoff = pd.Timestamp(cutoff_date)
    o = orders.merge(customers[['customer_id', 'customer_unique_id']],
                     on='customer_id', how='inner')
    o['order_purchase_timestamp'] = pd.to_datetime(o['order_purchase_timestamp'])
    o = o[o['order_purchase_timestamp'] <= cutoff]
    o = o[~o['order_status'].isin(['canceled', 'unavailable'])]
    merged = o.merge(order_items, on='order_id').merge(
        products[['product_id', 'product_category_name']], on='product_id',
    )
    cats = sorted(merged['product_category_name'].dropna().unique())
    cat_to_idx = {c: i+1 for i, c in enumerate(cats)}   # 0 = unknown
    out = np.zeros((len(customer_ids), top_k), dtype=np.int64)
    row_map = {c: i for i, c in enumerate(customer_ids)}
    for cuid, grp in merged.groupby('customer_unique_id', sort=False):
        if cuid not in row_map: continue
        counts = grp['product_category_name'].value_counts().head(top_k)
        for i, c in enumerate(counts.index):
            out[row_map[cuid], i] = cat_to_idx.get(c, 0)
    return out, len(cat_to_idx) + 1


# Build features
cust_feats = build_customer_features(tables['orders'], tables['order_items'],
                                     tables['customers'], cutoff)
assert (cust_feats['recency_days'] >= 0).all(), 'NEGATIVE RECENCY = leak'
print(f'customer features:   {cust_feats.shape}')
print(f'feature cols:        {list(cust_feats.columns)}')

seq_data = build_interaction_sequences(tables['orders'], tables['order_items'],
                                       tables['customers'], cutoff)
print(f'\nsequences:           n={len(seq_data.customer_ids)}, L={seq_data.product_ids.shape[1]}')
print(f'product vocab size:  {seq_data.product_id_vocab_size}')
print('\n✓ Feature invariants hold')

## 5. FT-Transformer — Customer Tower

Tokenizes each numerical feature → self-attention captures feature interactions.
(Gorishniy et al. 2021)

In [ ]:
class NumericalFeatureTokenizer(nn.Module):
    def __init__(self, n_features, d_token):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(n_features, d_token))
        self.bias = nn.Parameter(torch.empty(n_features, d_token))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        nn.init.zeros_(self.bias)
    def forward(self, x):
        return x.unsqueeze(-1) * self.weight + self.bias


class CLSToken(nn.Module):
    def __init__(self, d_token):
        super().__init__()
        self.cls = nn.Parameter(torch.empty(1, 1, d_token))
        nn.init.trunc_normal_(self.cls, std=0.02)
    def forward(self, x):
        return torch.cat([self.cls.expand(x.size(0), -1, -1), x], dim=1)


class FTBlock(nn.Module):
    def __init__(self, d, n_heads, drop=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=drop, batch_first=True)
        self.norm2 = nn.LayerNorm(d)
        h = int(d * 4/3)
        self.ffn = nn.Sequential(nn.Linear(d, h), nn.GELU(), nn.Dropout(drop), nn.Linear(h, d))
    def forward(self, x):
        h = self.norm1(x); a, _ = self.attn(h, h, h, need_weights=False); x = x + a
        x = x + self.ffn(self.norm2(x))
        return x


class FTTransformer(nn.Module):
    def __init__(self, n_features, d_token=64, n_blocks=3, n_heads=8, drop=0.1):
        super().__init__()
        self.tok = NumericalFeatureTokenizer(n_features, d_token)
        self.cls = CLSToken(d_token)
        self.blocks = nn.ModuleList([FTBlock(d_token, n_heads, drop) for _ in range(n_blocks)])
        self.norm = nn.LayerNorm(d_token)
    def forward(self, x):
        h = self.cls(self.tok(x))
        for b in self.blocks: h = b(h)
        return self.norm(h)


# Smoke test
_ft = FTTransformer(n_features=6, d_token=32, n_blocks=2, n_heads=4)
_out = _ft(torch.randn(4, 6))
assert _out.shape == (4, 7, 32)
print(f'✓ FT-Transformer: (4,6) -> {tuple(_out.shape)}')
print(f'  params: {sum(p.numel() for p in _ft.parameters()):,}')

## 6. Perceiver IO hub + Output-query heads

Small learned latent array cross-attends to inputs, then self-attends among latents.
Each task head is an output query that cross-attends to the latents.
(Jaegle et al. 2021)

In [ ]:
class CrossAttention(nn.Module):
    def __init__(self, q_dim, kv_dim, n_heads, drop=0.0):
        super().__init__()
        self.q_norm = nn.LayerNorm(q_dim)
        self.kv_norm = nn.LayerNorm(kv_dim)
        self.kv_proj = nn.Linear(kv_dim, q_dim)
        self.attn = nn.MultiheadAttention(q_dim, n_heads, dropout=drop, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(q_dim, q_dim*2), nn.GELU(),
                                 nn.Dropout(drop), nn.Linear(q_dim*2, q_dim))
        self.out_norm = nn.LayerNorm(q_dim)
    def forward(self, q, kv, key_padding_mask=None):
        qn = self.q_norm(q)
        kvn = self.kv_proj(self.kv_norm(kv))
        a, _ = self.attn(qn, kvn, kvn, key_padding_mask=key_padding_mask, need_weights=False)
        q = q + a
        q = q + self.ffn(self.out_norm(q))
        return q


class SelfAttnBlock(nn.Module):
    def __init__(self, d, n_heads, drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=drop, batch_first=True)
        self.norm2 = nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, d*4), nn.GELU(),
                                 nn.Dropout(drop), nn.Linear(d*4, d))
    def forward(self, x):
        h = self.norm1(x); a, _ = self.attn(h, h, h, need_weights=False); x = x + a
        x = x + self.ffn(self.norm2(x)); return x


class PerceiverIOHub(nn.Module):
    def __init__(self, num_latents=64, latent_dim=128, input_dim=128,
                 n_cross_heads=4, n_self_heads=8, n_sa_per_layer=2, n_layers=4, drop=0.1):
        super().__init__()
        self.latents = nn.Parameter(torch.empty(num_latents, latent_dim))
        nn.init.trunc_normal_(self.latents, std=0.02)
        self.cross = nn.ModuleList([
            CrossAttention(latent_dim, input_dim, n_cross_heads, drop) for _ in range(n_layers)
        ])
        self.sa_stacks = nn.ModuleList([
            nn.ModuleList([SelfAttnBlock(latent_dim, n_self_heads, drop) for _ in range(n_sa_per_layer)])
            for _ in range(n_layers)
        ])
    def forward(self, inputs, input_mask=None):
        B = inputs.size(0)
        lat = self.latents.unsqueeze(0).expand(B, -1, -1)
        kp_mask = (~input_mask) if input_mask is not None else None
        for cr, sa_stack in zip(self.cross, self.sa_stacks):
            lat = cr(lat, inputs, key_padding_mask=kp_mask)
            for sa in sa_stack: lat = sa(lat)
        return lat


class OutputQueryHead(nn.Module):
    """One learned query per task → cross-attends to latents → MLP to output dim."""
    def __init__(self, latent_dim, query_dim, output_dim, n_heads=4, hidden=128, drop=0.1):
        super().__init__()
        self.query = nn.Parameter(torch.empty(1, 1, query_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.cross = CrossAttention(query_dim, latent_dim, n_heads, drop)
        self.mlp = nn.Sequential(nn.LayerNorm(query_dim), nn.Linear(query_dim, hidden),
                                 nn.GELU(), nn.Dropout(drop), nn.Linear(hidden, output_dim))
    def forward(self, lat):
        B = lat.size(0)
        q = self.query.expand(B, -1, -1)
        h = self.cross(q, lat)
        return self.mlp(h.squeeze(1))


# Smoke test
_hub = PerceiverIOHub(num_latents=16, latent_dim=64, input_dim=64, n_layers=2)
_lat = _hub(torch.randn(4, 30, 64))
assert _lat.shape == (4, 16, 64)
_h = OutputQueryHead(64, 64, 1)
_pred = _h(_lat)
assert _pred.shape == (4, 1)
print(f'✓ Perceiver hub: latents {tuple(_lat.shape)}')
print(f'✓ OutputQueryHead: {tuple(_pred.shape)}')

## 7. Three input towers

Each tower projects to the Perceiver's input_dim so the hub can cross-attend uniformly.

In [ ]:
class CustomerTower(nn.Module):
    def __init__(self, n_tabular_features, d_token=64, n_blocks=3, n_heads=8, output_dim=128, drop=0.1):
        super().__init__()
        self.ft = FTTransformer(n_tabular_features, d_token, n_blocks, n_heads, drop)
        self.proj = nn.Linear(d_token, output_dim)
    def forward(self, x):
        return self.proj(self.ft(x))


class ProductInteractionTower(nn.Module):
    """Sequence of (product_emb + time_emb) tokens, one per interaction."""
    def __init__(self, product_vocab_size, output_dim=128, drop=0.1):
        super().__init__()
        self.product_emb = nn.Embedding(product_vocab_size, output_dim, padding_idx=0)
        self.time_mlp = nn.Sequential(nn.Linear(3, output_dim), nn.GELU(),
                                      nn.Linear(output_dim, output_dim))
        self.norm = nn.LayerNorm(output_dim)
        self.drop = nn.Dropout(drop)
    def forward(self, pids, time_deltas, mask):
        p = self.product_emb(pids)
        td = time_deltas.clamp(min=0.0).unsqueeze(-1)
        tf = torch.cat([torch.log1p(td), td/30.0, td/365.0], dim=-1)
        t = self.time_mlp(tf)
        h = self.drop(self.norm(p + t))
        return h * mask.unsqueeze(-1).float()


class ProductCatalogTower(nn.Module):
    """Aggregated category embedding — one token per customer."""
    def __init__(self, n_categories, output_dim=128):
        super().__init__()
        self.cat_emb = nn.Embedding(n_categories + 1, output_dim, padding_idx=0)
        self.norm = nn.LayerNorm(output_dim)
    def forward(self, cat_ids):
        return self.norm(self.cat_emb(cat_ids).mean(dim=1, keepdim=True))


print('✓ Three towers defined')

## 8. Full EvoCRM model

Wires towers → fuse → Perceiver hub → three output-query heads.
Multi-task loss uses uncertainty weighting (Kendall et al. 2018).

In [ ]:
TASK_SPECS = [
    ('churn',    'binary',     1),
    ('category', 'multiclass', N_CATEGORY_CLASSES),
    ('clv',      'regression', 1),
]


class EvoCRM(nn.Module):
    def __init__(self, n_tabular, product_vocab_size, n_categories, task_specs=TASK_SPECS):
        super().__init__()
        D = PERCEIVER_INPUT_DIM
        self.customer_tower = CustomerTower(
            n_tabular_features=n_tabular, d_token=FT_D_TOKEN,
            n_blocks=FT_N_BLOCKS, n_heads=FT_N_HEADS, output_dim=D,
        )
        self.interaction_tower = ProductInteractionTower(product_vocab_size, D)
        self.catalog_tower = ProductCatalogTower(n_categories, D)
        self.hub = PerceiverIOHub(
            num_latents=PERCEIVER_NUM_LATENTS, latent_dim=PERCEIVER_LATENT_DIM,
            input_dim=D, n_layers=PERCEIVER_N_LAYERS,
        )
        self.heads = nn.ModuleDict()
        self.task_types = {}
        for name, ttype, out_dim in task_specs:
            self.heads[name] = OutputQueryHead(
                latent_dim=PERCEIVER_LATENT_DIM, query_dim=PERCEIVER_LATENT_DIM,
                output_dim=out_dim,
            )
            self.task_types[name] = ttype
        # Uncertainty weighting (Kendall 2018)
        self.log_vars = nn.ParameterDict({
            name: nn.Parameter(torch.zeros(1)) for name, _, _ in task_specs
        })

    def forward(self, tabular, seq_pids, seq_dt, seq_mask, cat_ids):
        ct = self.customer_tower(tabular)                       # (B, F+1, D)
        it = self.interaction_tower(seq_pids, seq_dt, seq_mask) # (B, L, D)
        cg = self.catalog_tower(cat_ids)                        # (B, 1, D)
        B = ct.size(0); dev = tabular.device
        ct_mask = torch.ones(B, ct.size(1), dtype=torch.bool, device=dev)
        cg_mask = torch.ones(B, cg.size(1), dtype=torch.bool, device=dev)
        fused = torch.cat([ct, it, cg], dim=1)
        fused_mask = torch.cat([ct_mask, seq_mask, cg_mask], dim=1)
        lat = self.hub(fused, input_mask=fused_mask)
        return {name: head(lat) for name, head in self.heads.items()}

    def compute_loss(self, outputs, targets, valid_masks):
        losses = {}
        total = 0.0
        for name, ttype in self.task_types.items():
            pred = outputs[name]; tgt = targets[name]; vm = valid_masks[name]
            if vm.sum() == 0:
                losses[name] = torch.tensor(0.0, device=pred.device); continue
            pv, tv = pred[vm], tgt[vm]
            if ttype == 'binary':
                l = nn.functional.binary_cross_entropy_with_logits(
                    pv.squeeze(-1), tv.float(), reduction='mean')
            elif ttype == 'multiclass':
                l = nn.functional.cross_entropy(pv, tv.long())
            elif ttype == 'regression':
                ps = pv.squeeze(-1)
                ts = torch.log1p(tv.clamp(min=0.0).float())
                l = nn.functional.huber_loss(ps, ts, delta=1.0)
            losses[name] = l
            lv = self.log_vars[name]
            total = total + (0.5 * torch.exp(-lv) * l + 0.5 * lv).squeeze()
        losses['total'] = total
        return losses


# Smoke test
_m = EvoCRM(n_tabular=6, product_vocab_size=100, n_categories=12)
_out = _m(
    tabular=torch.randn(4, 6),
    seq_pids=torch.randint(0, 100, (4, 20)),
    seq_dt=torch.rand(4, 20) * 300,
    seq_mask=torch.ones(4, 20, dtype=torch.bool),
    cat_ids=torch.randint(0, 12, (4, 3)),
)
for n, p in _out.items():
    print(f'  {n:>10s}: {tuple(p.shape)}')
print(f'\n✓ EvoCRM forward pass — total params: {sum(p.numel() for p in _m.parameters()):,}')

## 9. PyTorch Dataset

In [ ]:
class EvoCRMDataset(Dataset):
    def __init__(self, tabular, seq_pids, seq_dt, seq_mask, cat_ids, labels, valid_masks):
        self.tabular = tabular.astype(np.float32)
        self.seq_pids = seq_pids.astype(np.int64)
        self.seq_dt = seq_dt.astype(np.float32)
        self.seq_mask = seq_mask.astype(bool)
        self.cat_ids = cat_ids.astype(np.int64)
        self.labels = labels
        self.valid_masks = {k: v.astype(bool) for k, v in valid_masks.items()}
    def __len__(self): return len(self.tabular)
    def __getitem__(self, i):
        item = {
            'tabular':  torch.from_numpy(self.tabular[i]),
            'seq_pids': torch.from_numpy(self.seq_pids[i]),
            'seq_dt':   torch.from_numpy(self.seq_dt[i]),
            'seq_mask': torch.from_numpy(self.seq_mask[i]),
            'cat_ids':  torch.from_numpy(self.cat_ids[i]),
        }
        for name, arr in self.labels.items():
            item[f'label_{name}'] = torch.tensor(arr[i])
        for name, m in self.valid_masks.items():
            item[f'valid_{name}'] = torch.tensor(m[i])
        return item


# Assemble full feature matrix aligned with labels
common_ids = [c for c in labels['customer_unique_id'].values if c in cust_feats.index]
labels_aligned = labels.set_index('customer_unique_id').loc[common_ids].reset_index()

# Tabular
tabular_mat = cust_feats.loc[common_ids].to_numpy(dtype=np.float32)

# Sequences — align to common_ids
seq_map = {c: i for i, c in enumerate(seq_data.customer_ids)}
empty_p = np.zeros(seq_data.product_ids.shape[1], dtype=np.int64)
empty_d = np.zeros(seq_data.timestamps_delta.shape[1], dtype=np.float32)
empty_m = np.zeros(seq_data.mask.shape[1], dtype=bool)
seq_pids_m, seq_dt_m, seq_mask_m = [], [], []
for cuid in common_ids:
    j = seq_map.get(cuid)
    if j is None:
        seq_pids_m.append(empty_p); seq_dt_m.append(empty_d); seq_mask_m.append(empty_m)
    else:
        seq_pids_m.append(seq_data.product_ids[j])
        seq_dt_m.append(seq_data.timestamps_delta[j])
        seq_mask_m.append(seq_data.mask[j])
seq_pids_arr = np.stack(seq_pids_m)
seq_dt_arr = np.stack(seq_dt_m)
seq_mask_arr = np.stack(seq_mask_m)

# Category ids
cat_ids_arr, n_categories_total = build_customer_category_ids(
    tables['orders'], tables['order_items'], tables['products'],
    tables['customers'], cutoff, common_ids, top_k=3,
)

# Random train/val split (temporal safety already enforced by cutoff)
rng = np.random.default_rng(SEED)
idx = np.arange(len(common_ids)); rng.shuffle(idx)
n_val = int(len(idx) * VAL_FRACTION)
val_idx, trn_idx = idx[:n_val], idx[n_val:]

def make_ds(rows):
    cat_vals = labels_aligned['category'].values[rows]
    return EvoCRMDataset(
        tabular=tabular_mat[rows],
        seq_pids=seq_pids_arr[rows],
        seq_dt=seq_dt_arr[rows],
        seq_mask=seq_mask_arr[rows],
        cat_ids=cat_ids_arr[rows],
        labels={
            'churn':    labels_aligned['churn'].values[rows].astype(np.int64),
            'category': np.where(cat_vals >= 0, cat_vals, 0).astype(np.int64),
            'clv':      labels_aligned['clv'].values[rows].astype(np.float32),
        },
        valid_masks={
            'churn':    labels_aligned['churn_valid'].values[rows],
            'category': labels_aligned['category_valid'].values[rows],
            'clv':      labels_aligned['clv_valid'].values[rows],
        },
    )

train_ds = make_ds(trn_idx)
val_ds = make_ds(val_idx)
print(f'train: {len(train_ds):,} | val: {len(val_ds):,}')
print(f'n_categories (catalog tower): {n_categories_total}')

## 10. Training loop

In [ ]:
def _eval_total(model, loader):
    model.eval(); tot, n = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            b = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(b['tabular'], b['seq_pids'], b['seq_dt'], b['seq_mask'], b['cat_ids'])
            tgt = {n: b[f'label_{n}'] for n in model.task_types}
            vm = {n: b[f'valid_{n}'] for n in model.task_types}
            losses = model.compute_loss(out, tgt, vm)
            bs = b['tabular'].size(0)
            tot += float(losses['total'].item()) * bs; n += bs
    return tot / max(n, 1)


def train_model(model, train_ds, val_ds, num_epochs=NUM_EPOCHS, verbose=True):
    model.to(DEVICE)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    opt = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = max(1, len(train_loader) * num_epochs)
    sched = CosineAnnealingLR(opt, T_max=total_steps)

    history = {'train': [], 'val': []}
    best, patience_ctr = float('inf'), 0
    for ep in range(num_epochs):
        model.train(); tot, n = 0.0, 0
        for batch in train_loader:
            b = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(b['tabular'], b['seq_pids'], b['seq_dt'], b['seq_mask'], b['cat_ids'])
            tgt = {name: b[f'label_{name}'] for name in model.task_types}
            vm = {name: b[f'valid_{name}'] for name in model.task_types}
            losses = model.compute_loss(out, tgt, vm)
            loss = losses['total']
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            opt.step(); sched.step()
            bs = b['tabular'].size(0)
            tot += float(loss.item()) * bs; n += bs
        tr = tot / max(n, 1); va = _eval_total(model, val_loader)
        history['train'].append(tr); history['val'].append(va)
        if va < best - 1e-4: best, patience_ctr = va, 0
        else: patience_ctr += 1
        if verbose:
            print(f'  epoch {ep+1:3d} | train={tr:.4f} | val={va:.4f}')
        if patience_ctr >= PATIENCE:
            if verbose: print(f'  early stop at epoch {ep+1}'); break
    return history

print('✓ Training loop defined')

## 11. Evaluation

In [ ]:
def evaluate(model, dataset, batch_size=256):
    model.to(DEVICE).eval()
    loader = DataLoader(dataset, batch_size=batch_size)
    preds = {n: [] for n in model.task_types}
    targets = {n: [] for n in model.task_types}
    valid = {n: [] for n in model.task_types}
    with torch.no_grad():
        for batch in loader:
            b = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(b['tabular'], b['seq_pids'], b['seq_dt'], b['seq_mask'], b['cat_ids'])
            for n in model.task_types:
                preds[n].append(out[n].cpu().numpy())
                targets[n].append(b[f'label_{n}'].cpu().numpy())
                valid[n].append(b[f'valid_{n}'].cpu().numpy())

    results = {}
    for name, ttype in model.task_types.items():
        p = np.concatenate(preds[name]); t = np.concatenate(targets[name])
        v = np.concatenate(valid[name]).astype(bool)
        if v.sum() == 0: results[name] = {'n_valid': 0}; continue
        pv, tv = p[v], t[v]
        if ttype == 'binary':
            logits = pv.squeeze(-1); probs = 1.0/(1.0 + np.exp(-logits))
            pred_cls = (probs > 0.5).astype(int)
            try: auc = roc_auc_score(tv, probs)
            except ValueError: auc = float('nan')
            results[name] = {
                'n_valid': int(v.sum()), 'auc': float(auc),
                'f1': float(f1_score(tv, pred_cls, zero_division=0)),
                'accuracy': float(accuracy_score(tv, pred_cls)),
                'pos_rate': float(tv.mean()),
            }
        elif ttype == 'multiclass':
            pred_cls = pv.argmax(axis=-1)
            results[name] = {
                'n_valid': int(v.sum()),
                'accuracy': float(accuracy_score(tv, pred_cls)),
                'f1_macro': float(f1_score(tv, pred_cls, average='macro', zero_division=0)),
                'f1_weighted': float(f1_score(tv, pred_cls, average='weighted', zero_division=0)),
            }
        elif ttype == 'regression':
            pred_clv = np.expm1(pv.squeeze(-1)).clip(min=0)
            results[name] = {
                'n_valid': int(v.sum()),
                'rmse': float(np.sqrt(mean_squared_error(tv, pred_clv))),
                'mae': float(mean_absolute_error(tv, pred_clv)),
                'target_mean': float(tv.mean()),
            }
    return results

print('✓ Evaluation defined')

## 12. Specialist baselines

One classical model per task on the same tabular features — the apples-to-apples comparison.

In [ ]:
def run_baselines(train_ds, val_ds):
    X_tr, X_va = train_ds.tabular, val_ds.tabular
    results = {}
    for name, ttype in [('churn','binary'), ('category','multiclass'), ('clv','regression')]:
        vm_tr = train_ds.valid_masks[name]
        vm_va = val_ds.valid_masks[name]
        y_tr = train_ds.labels[name][vm_tr]; y_va = val_ds.labels[name][vm_va]
        Xt, Xv = X_tr[vm_tr], X_va[vm_va]
        if len(y_tr) < 20 or len(y_va) < 5:
            results[name] = {'error': 'insufficient data'}; continue
        task_r = {}
        if ttype == 'binary':
            for mname, clf in [
                ('gbm', GradientBoostingClassifier(random_state=SEED, n_estimators=100)),
                ('rf', RandomForestClassifier(random_state=SEED, n_estimators=200)),
                ('logreg', LogisticRegression(max_iter=500, random_state=SEED)),
            ]:
                try:
                    clf.fit(Xt, y_tr)
                    proba = clf.predict_proba(Xv)[:, 1]
                    pred = (proba > 0.5).astype(int)
                    task_r[mname] = {
                        'auc': float(roc_auc_score(y_va, proba)) if len(np.unique(y_va))>1 else float('nan'),
                        'f1': float(f1_score(y_va, pred, zero_division=0)),
                        'accuracy': float(accuracy_score(y_va, pred)),
                    }
                except Exception as e: task_r[mname] = {'error': str(e)}
        elif ttype == 'multiclass':
            for mname, clf in [
                ('gbm', GradientBoostingClassifier(random_state=SEED, n_estimators=100)),
                ('rf', RandomForestClassifier(random_state=SEED, n_estimators=200)),
                ('logreg', LogisticRegression(max_iter=500, random_state=SEED)),
            ]:
                try:
                    clf.fit(Xt, y_tr); pred = clf.predict(Xv)
                    task_r[mname] = {
                        'accuracy': float(accuracy_score(y_va, pred)),
                        'f1_macro': float(f1_score(y_va, pred, average='macro', zero_division=0)),
                        'f1_weighted': float(f1_score(y_va, pred, average='weighted', zero_division=0)),
                    }
                except Exception as e: task_r[mname] = {'error': str(e)}
        elif ttype == 'regression':
            y_tr_log = np.log1p(np.clip(y_tr, 0, None))
            for mname, reg in [
                ('gbm', GradientBoostingRegressor(random_state=SEED, n_estimators=100)),
                ('rf', RandomForestRegressor(random_state=SEED, n_estimators=200)),
            ]:
                try:
                    reg.fit(Xt, y_tr_log)
                    pred = np.expm1(reg.predict(Xv)).clip(min=0)
                    task_r[mname] = {
                        'rmse': float(np.sqrt(mean_squared_error(y_va, pred))),
                        'mae': float(mean_absolute_error(y_va, pred)),
                    }
                except Exception as e: task_r[mname] = {'error': str(e)}
        results[name] = task_r
    return results

print('✓ Baselines defined')

## 13. LoRA injection (self-evolving mechanism)

After initial training, freeze the base model and inject rank-8 LoRA adapters into attention `out_proj` layers. Subsequent updates modify only ~1–5% of parameters — the overnight-adaptation story.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base_linear, rank=8, alpha=16, drop=0.05):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters(): p.requires_grad = False
        in_f, out_f = base_linear.in_features, base_linear.out_features
        self.lora_A = nn.Parameter(torch.empty(rank, in_f))
        self.lora_B = nn.Parameter(torch.zeros(out_f, rank))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.scale = alpha / rank
        self.drop = nn.Dropout(drop) if drop > 0 else nn.Identity()
    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.lora_A.T @ self.lora_B.T) * self.scale


def inject_lora_into_attention(model, rank=LORA_RANK, alpha=LORA_ALPHA, drop=0.05):
    wrapped = 0
    for _, parent in list(model.named_modules()):
        for attr_name, child in list(parent.named_children()):
            if isinstance(child, nn.MultiheadAttention):
                if hasattr(child, 'out_proj') and isinstance(child.out_proj, nn.Linear):
                    child.out_proj = LoRALinear(child.out_proj, rank, alpha, drop); wrapped += 1
    return wrapped


def freeze_non_lora(model):
    for p in model.parameters(): p.requires_grad = False
    n_tr = 0
    for m in model.modules():
        if isinstance(m, LoRALinear):
            m.lora_A.requires_grad = True; m.lora_B.requires_grad = True
            n_tr += m.lora_A.numel() + m.lora_B.numel()
    return n_tr


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {'total': total, 'trainable': trainable, 'trainable_pct': 100.0 * trainable / max(total, 1)}

print('✓ LoRA helpers defined')

## 14. End-to-end run

Build model → train → evaluate → run baselines → LoRA pass.

In [ ]:
# Build model
model = EvoCRM(
    n_tabular=tabular_mat.shape[1],
    product_vocab_size=seq_data.product_id_vocab_size,
    n_categories=n_categories_total,
)
print(f'data source: {DATA_SOURCE}')
print(f'model params: {sum(p.numel() for p in model.parameters()):,}\n')

# Train
print('=== TRAINING ===')
history = train_model(model, train_ds, val_ds, num_epochs=NUM_EPOCHS, verbose=True)

# Evaluate
print('\n=== EVOCRM METRICS (val) ===')
evocrm_val_metrics = evaluate(model, val_ds)
for task, m in evocrm_val_metrics.items():
    print(f'  {task}: {m}')

In [ ]:
# Baselines
print('=== SPECIALIST BASELINES (val) ===')
baseline_metrics = run_baselines(train_ds, val_ds)
for task, methods in baseline_metrics.items():
    print(f'\n  {task}:')
    for mname, vals in methods.items():
        print(f'    {mname}: {vals}')

In [ ]:
# LoRA pass — freeze base, inject adapters, fine-tune for a few epochs
print('=== LoRA SELF-EVOLUTION PASS ===')
lora_model = copy.deepcopy(model).to(DEVICE)
before = count_parameters(lora_model)
n_wrapped = inject_lora_into_attention(lora_model)
# Re-move to device (new modules were added on CPU)
lora_model = lora_model.to(DEVICE)
freeze_non_lora(lora_model)
after = count_parameters(lora_model)
print(f'  wrapped {n_wrapped} attention projections')
print(f'  before LoRA: {before["total"]:,} total, {before["trainable"]:,} trainable')
print(f'  after LoRA:  {after["total"]:,} total, {after["trainable"]:,} trainable '
      f'({after["trainable_pct"]:.2f}%)')
print(f'  parameter reduction: {100 - after["trainable_pct"]:.1f}%')

# Brief LoRA-only training pass (simulates overnight adaptation)
print('\n  training only LoRA adapters for 3 epochs...')
_ = train_model(lora_model, train_ds, val_ds, num_epochs=3, verbose=True)

print('\n=== LoRA-ADAPTED METRICS (val) ===')
lora_val_metrics = evaluate(lora_model, val_ds)
for task, m in lora_val_metrics.items():
    print(f'  {task}: {m}')

## 15. Diagnostics — did we actually avoid leakage?

Healthy churn AUC: **0.70–0.85**. AUC > 0.95 is a leak warning.

In [ ]:
churn_auc = evocrm_val_metrics['churn'].get('auc', 0.0)
print(f'\nEvoCRM churn AUC: {churn_auc:.4f}')
if churn_auc > 0.95:
    print('⚠️  WARNING: AUC > 0.95 — likely label/feature leakage. Re-audit cutoff discipline.')
elif churn_auc < 0.55:
    print('ℹ️  AUC near chance — try more epochs, larger data, or verify label quality.')
else:
    print('✓ AUC in healthy range (0.55–0.95).')

# Compare EvoCRM vs best baseline on each task
print('\n=== EVOCRM vs BEST BASELINE ===')
for task in ['churn', 'category', 'clv']:
    evo = evocrm_val_metrics.get(task, {})
    bl = baseline_metrics.get(task, {})
    if task == 'churn':
        metric = 'auc'; better = 'higher'
    elif task == 'category':
        metric = 'f1_macro'; better = 'higher'
    else:
        metric = 'rmse'; better = 'lower'
    evo_v = evo.get(metric, float('nan'))
    bl_scores = [v[metric] for v in bl.values() if isinstance(v, dict) and metric in v]
    if not bl_scores: continue
    bl_best = max(bl_scores) if better == 'higher' else min(bl_scores)
    bl_best_method = max(bl.items(), key=lambda kv: kv[1].get(metric, -np.inf)) \
        if better=='higher' else min(bl.items(), key=lambda kv: kv[1].get(metric, np.inf))
    delta = evo_v - bl_best if better == 'higher' else bl_best - evo_v
    sign = '+' if delta >= 0 else ''
    print(f'  {task:<10s} {metric:<12s}  EvoCRM={evo_v:.4f}  best_baseline={bl_best:.4f} ({bl_best_method[0]})  Δ={sign}{delta:.4f}')

In [ ]:
# Save all results to disk for later analysis
results = {
    'data_source': DATA_SOURCE,
    'cutoff_date': str(cutoff),
    'n_train': len(train_ds),
    'n_val': len(val_ds),
    'churn_rate_val': float(labels_aligned.iloc[val_idx]['churn'].mean()),
    'model_params': sum(p.numel() for p in model.parameters()),
    'lora_trainable_pct': after['trainable_pct'],
    'train_history': history,
    'evocrm_val_metrics': evocrm_val_metrics,
    'baseline_metrics': baseline_metrics,
    'lora_val_metrics': lora_val_metrics,
}
with open('evocrm_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print('✓ saved evocrm_results.json')
print(f'  {len(json.dumps(results, default=str)):,} bytes')

---

## Done.

**What you got:**
- Full EvoCRM model (Perceiver IO + FT-Transformer + output-query heads) trained and evaluated
- Three tasks: binary churn, multi-class category, regression CLV
- Classical specialist baselines for apples-to-apples comparison
- LoRA injection + brief adaptation pass (the self-evolving mechanism)
- Leak invariant checks throughout
- Results saved to `evocrm_results.json`

**To use real Olist:**
1. Download from https://www.kaggle.com/olistbr/brazilian-ecommerce
2. Extract CSVs into `./olist_raw/` (or change `OLIST_DIR` in cell 2)
3. Re-run top-to-bottom

**If churn AUC > 0.95:** something is leaking. Check every source-table filter uses `<= cutoff_date`, and every label uses `> cutoff_date`. The two sets must be disjoint.